In [4]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime
from tqdm.auto import tqdm
import pandas as pd

# 1. Setup Client
API_KEY = "PKNPLUUHRK5EF67XWNIKDYP5LX"
SECRET_KEY = "B3jdCgy9eUJWqrq6h7Gi34QpPxHiA5d7HMxU7rkHY95E"
client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

symbol = "SPY"
start_date = datetime(2015, 1, 1)
end_date = datetime(2025, 12, 31)
time_frame = TimeFrame.Minute
output_file = rf"G:\Mi unidad\2026\Tesis\Códigos\Tools\sp500_{start_date.strftime('%Y')}_to_{end_date.strftime('%Y')}_{time_frame.value}.csv"

# 2. Build monthly chunks
chunks = []
chunk_start = pd.Timestamp(start_date)
final_end = pd.Timestamp(end_date)

while chunk_start < final_end:
    chunk_end = min(chunk_start + pd.DateOffset(months=1), final_end)
    chunks.append((chunk_start.to_pydatetime(), chunk_end.to_pydatetime()))
    chunk_start = chunk_end

# 3. Stream directly to CSV (faster than concat + single huge write)
header_written = False
last_written_ts = None
total_rows_written = 0

with tqdm(total=len(chunks), desc=f"Downloading {symbol}", unit="chunk") as pbar:
    for i, (current_start, current_end) in enumerate(chunks, start=1):
        request_params = StockBarsRequest(
            symbol_or_symbols=[symbol],
            timeframe=time_frame,
            start=current_start,
            end=current_end,
        )
        bars = client.get_stock_bars(request_params)
        part = bars.df

        if not part.empty:
            # Ensure deterministic order and avoid duplicate boundary rows between chunks.
            part = part.sort_index()
            if last_written_ts is not None:
                part = part[part.index > last_written_ts]

            if not part.empty:
                part.to_csv(
                    output_file,
                    mode='w' if not header_written else 'a',
                    header=not header_written
                )
                header_written = True
                last_written_ts = part.index.max()
                total_rows_written += len(part)

        pbar.set_postfix(chunk=f"{i}/{len(chunks)}", rows=f"{total_rows_written:,}")
        pbar.update(1)

# Keep a DataFrame in memory only if needed later in notebook
if header_written:
    df = pd.read_csv(output_file, parse_dates=['timestamp'])
else:
    df = pd.DataFrame()
    df.to_csv(output_file, index=False)

print(f"Download complete. File saved as {output_file}. Rows: {total_rows_written:,}")

Download complete. File saved as G:\Mi unidad\2026\Tesis\Códigos\Tools\sp500_2015_to_2025_1Min.csv. Rows: 1,985,063


In [14]:
import os
print("CWD:", os.getcwd())
print("Absolute CSV path:", os.path.abspath(output_file))
print("Exists?", os.path.exists(os.path.abspath(output_file)))


CWD: C:\Users\migue\AppData\Local\Programs\Microsoft VS Code
Absolute CSV path: C:\Users\migue\AppData\Local\Programs\Microsoft VS Code\sp500_2025_to_2025_1Min.csv
Exists? True


In [ ]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockTradesRequest
from datetime import datetime, timedelta
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import os
import time
from pathlib import Path

# ============================================================================
# FULLY OPTIMIZED TRANSACTIONS DOWNLOADER
# ============================================================================

class OptimizedTradesDownloader:
    """
    High-performance transactions downloader with caching, retries, and parallelization.
    """
    
    def __init__(self, api_key: str, secret_key: str, max_workers: int = 3):
        self.client = StockHistoricalDataClient(api_key, secret_key)
        self.max_workers = max_workers  # Conservative to respect rate limits
        self.session_stats = {
            'downloaded': 0,
            'cached': 0,
            'failed': 0,
            'start_time': time.time()
        }
    
    def _calculate_optimal_chunk_size(self, date_range_days: int) -> str:
        """Auto-scale chunk size based on date range."""
        if date_range_days <= 5:
            return 'daily'
        elif date_range_days <= 60:
            return 'weekly'
        elif date_range_days <= 365:
            return 'biweekly'
        else:
            return 'monthly'
    
    def _build_chunks(self, start_date: datetime, end_date: datetime, chunk_type: str = 'auto') -> list:
        """Build date chunks with automatic sizing."""
        if chunk_type == 'auto':
            days = (pd.Timestamp(end_date) - pd.Timestamp(start_date)).days
            chunk_type = self._calculate_optimal_chunk_size(days)
        
        chunks = []
        chunk_start = pd.Timestamp(start_date)
        final_end = pd.Timestamp(end_date)
        
        offset_map = {
            'daily': timedelta(days=1),
            'weekly': timedelta(weeks=1),
            'biweekly': timedelta(weeks=2),
            'monthly': pd.DateOffset(months=1)
        }
        
        offset = offset_map.get(chunk_type, timedelta(days=1))
        
        while chunk_start < final_end:
            chunk_end = min(chunk_start + offset, final_end)
            chunks.append((chunk_start.to_pydatetime(), chunk_end.to_pydatetime()))
            chunk_start = chunk_end
        
        return chunks
    
    def _get_cached_dates(self, output_file: str) -> set:
        """Extract downloaded dates from existing CSV to avoid re-downloading."""
        if not os.path.exists(output_file):
            return set()
        
        try:
            df = pd.read_csv(output_file, usecols=['timestamp'], parse_dates=['timestamp'])
            return set(df['timestamp'].dt.date)
        except Exception:
            return set()
    
    def _download_chunk_with_retry(self, symbol: str, start: datetime, end: datetime, 
                                   max_retries: int = 3) -> tuple:
        """Download chunk with exponential backoff retry logic."""
        for attempt in range(max_retries):
            try:
                request_params = StockTradesRequest(
                    symbol_or_symbols=symbol,
                    start=start,
                    end=end,
                    feed='sip'
                )
                
                trades_response = self.client.get_stock_trades(request_params)
                trades_df = trades_response.df
                
                if not trades_df.empty:
                    trades_df = trades_df.reset_index(level=0, drop=True)
                    trades_df = trades_df.sort_index()
                    return (trades_df, None)
                return (pd.DataFrame(), None)
                
            except Exception as e:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                    time.sleep(wait_time)
                else:
                    return (None, str(e))
        
        return (None, "Max retries exceeded")
    
    def download(self, 
                symbol: str, 
                start_date: datetime, 
                end_date: datetime,
                output_file: str = None,
                feed: str = 'sip',
                chunk_type: str = 'auto',
                use_cache: bool = True) -> pd.DataFrame:
        """
        Main download function with full optimizations.
        
        Args:
            symbol: Stock ticker (e.g., 'SPY')
            start_date: Start date
            end_date: End date
            output_file: CSV output path (auto-generated if None)
            feed: 'sip' (full) or 'iex' (free tier)
            chunk_type: 'daily', 'weekly', 'biweekly', 'monthly', or 'auto'
            use_cache: Skip already-downloaded dates
        
        Returns:
            DataFrame with all transactions
        """
        if output_file is None:
            output_dir = Path("G:\\Mi unidad\\2026\\Tesis\\Códigos\\Tools")
            output_dir.mkdir(parents=True, exist_ok=True)
            output_file = str(output_dir / f"{symbol}_trades_{start_date.strftime('%Y%m%d')}_to_{end_date.strftime('%Y%m%d')}.csv")
        
        # Get chunks
        chunks = self._build_chunks(start_date, end_date, chunk_type)
        cached_dates = self._get_cached_dates(output_file) if use_cache else set()
        
        # Filter chunks to skip cached dates
        chunks_to_download = []
        for start, end in chunks:
            if use_cache and pd.Timestamp(start).date() in cached_dates:
                self.session_stats['cached'] += 1
            else:
                chunks_to_download.append((start, end))
        
        print(f"\n📊 Optimized Download Plan:")
        print(f"  Symbol: {symbol}")
        print(f"  Date range: {start_date.date()} to {end_date.date()} ({len(chunks)} chunks)")
        print(f"  Cached chunks: {self.session_stats['cached']} (skipping)")
        print(f"  Chunks to download: {len(chunks_to_download)}")
        print(f"  Workers: {self.max_workers} | Feed: {feed} | Chunk type: {chunk_type}\n")
        
        if not chunks_to_download:
            print("✓ All data already cached! Loading from file...")
            return pd.read_csv(output_file, parse_dates=['timestamp'])
        
        # Process chunks in parallel
        header_written = False
        last_written_ts = None
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {
                executor.submit(self._download_chunk_with_retry, symbol, start, end): (start, end)
                for start, end in chunks_to_download
            }
            
            with tqdm(total=len(chunks_to_download), desc=f"↓ {symbol} trades", unit="chunk") as pbar:
                for future in as_completed(futures):
                    start, end = futures[future]
                    trades_df, error = future.result()
                    
                    if error:
                        self.session_stats['failed'] += 1
                        pbar.set_postfix_str(f"❌ Error: {error[:30]}")
                    elif trades_df is not None and not trades_df.empty:
                        # Deduplication: remove rows already in file
                        if last_written_ts is not None:
                            trades_df = trades_df[trades_df.index > last_written_ts]
                        
                        if not trades_df.empty:
                            trades_df.to_csv(
                                output_file,
                                mode='w' if not header_written else 'a',
                                header=not header_written,
                                index_label='timestamp'
                            )
                            header_written = True
                            last_written_ts = trades_df.index.max()
                            self.session_stats['downloaded'] += len(trades_df)
                    
                    pbar.update(1)
                    pbar.set_postfix(trades=f"{self.session_stats['downloaded']:,}")
        
        # Load final result
        if os.path.exists(output_file):
            df_result = pd.read_csv(output_file, parse_dates=['timestamp'])
        else:
            df_result = pd.DataFrame()
        
        # Print summary
        elapsed = time.time() - self.session_stats['start_time']
        self._print_summary(output_file, df_result, elapsed)
        
        return df_result
    
    def _print_summary(self, output_file: str, df: pd.DataFrame, elapsed: float):
        """Print detailed download summary."""
        print(f"\n{'='*70}")
        print(f"✓ DOWNLOAD COMPLETE")
        print(f"{'='*70}")
        print(f"  File: {output_file}")
        print(f"  Size: {os.path.getsize(output_file) / (1024**2):.2f} MB" if os.path.exists(output_file) else "  File not found")
        print(f"\n  📈 Statistics:")
        print(f"    • Transactions: {self.session_stats['downloaded']:,}")
        print(f"    • From cache: {self.session_stats['cached']:,}")
        print(f"    • Failed chunks: {self.session_stats['failed']}")
        print(f"    • Total rows in file: {len(df):,}")
        
        if not df.empty:
            print(f"\n  🕐 Time range:")
            print(f"    • First: {df['timestamp'].min()}")
            print(f"    • Last: {df['timestamp'].max()}")
            print(f"    • Span: {(df['timestamp'].max() - df['timestamp'].min()).days} days")
            print(f"\n  📋 Columns: {', '.join(df.columns)}")
        
        print(f"\n  ⏱️  Elapsed: {elapsed:.1f}s")
        print(f"{'='*70}\n")


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

# Initialize downloader
API_KEY = "PKNPLUUHRK5EF67XWNIKDYP5LX"
SECRET_KEY = "B3jdCgy9eUJWqrq6h7Gi34QpPxHiA5d7HMxU7rkHY95E"
downloader = OptimizedTradesDownloader(API_KEY, SECRET_KEY, max_workers=3)

# Download with automatic optimization
symbol = "SPY"
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 1, 31)

df_trades = downloader.download(
    symbol=symbol,
    start_date=start_date,
    end_date=end_date,
    chunk_type='auto',      # Auto-scales: daily/weekly/biweekly/monthly
    use_cache=True,          # Skip already-downloaded dates
    feed='sip'               # Full market or 'iex' for free tier
)

# For large date ranges (e.g., 2015-2025), use explicit chunking:
# df_trades = downloader.download(
#     symbol="SPY",
#     start_date=datetime(2015, 1, 1),
#     end_date=datetime(2025, 12, 31),
#     chunk_type='monthly',  # Explicit for stability
#     use_cache=True
# )


In [ ]:
# ============================================================================
# ADVANCED: MULTI-SYMBOL BATCH DOWNLOADER
# ============================================================================

def batch_download_trades(symbols: list, start_date: datetime, end_date: datetime, 
                         feed: str = 'sip', max_workers: int = 3) -> dict:
    """
    Download trades for multiple symbols efficiently with shared client.
    
    Args:
        symbols: List of tickers (e.g., ['SPY', 'QQQ', 'IWM'])
        start_date: Start date
        end_date: End date
        feed: 'sip' or 'iex'
        max_workers: Parallel downloads per symbol
    
    Returns:
        Dictionary mapping symbol -> DataFrame
    """
    downloader = OptimizedTradesDownloader(API_KEY, SECRET_KEY, max_workers=max_workers)
    results = {}
    
    with tqdm(total=len(symbols), desc="Multi-symbol download", unit="symbol") as pbar:
        for symbol in symbols:
            try:
                df = downloader.download(
                    symbol=symbol,
                    start_date=start_date,
                    end_date=end_date,
                    chunk_type='auto',
                    use_cache=True,
                    feed=feed
                )
                results[symbol] = df
                pbar.update(1)
            except Exception as e:
                print(f"\n❌ Failed to download {symbol}: {e}")
                results[symbol] = None
                pbar.update(1)
    
    return results


# ============================================================================
# EXAMPLE: MULTI-SYMBOL DOWNLOAD
# ============================================================================

# Uncomment to download multiple symbols:
# symbols = ['SPY', 'QQQ', 'IWM']
# start = datetime(2024, 1, 1)
# end = datetime(2024, 1, 31)
# 
# trades_dict = batch_download_trades(symbols, start, end, feed='sip', max_workers=2)
# 
# # Access individual symbol data
# for symbol, df in trades_dict.items():
#     if df is not None:
#         print(f"\n{symbol}: {len(df):,} trades")
#     else:
#         print(f"\n{symbol}: Failed to download")


In [ ]:
# ============================================================================
# PERFORMANCE TIPS & UTILITIES
# ============================================================================

def estimate_download_time(start_date: datetime, end_date: datetime, 
                          avg_trades_per_day: int = 1_000_000) -> dict:
    """
    Estimate download time and data size for a date range.
    
    Args:
        start_date: Start date
        end_date: End date
        avg_trades_per_day: Estimated trades per day (SPY ~1-3M)
    
    Returns:
        Dictionary with estimates
    """
    days = (pd.Timestamp(end_date) - pd.Timestamp(start_date)).days
    total_trades = days * avg_trades_per_day
    size_mb = (total_trades * 0.0006)  # ~600 bytes per row
    
    # Estimates: 3 parallel workers, ~5s per chunk, 50s for overhead
    chunks = days if days <= 5 else (days // 7 if days <= 365 else (days // 14 if days <= 730 else days // 30))
    time_per_chunk = 5  # seconds
    parallel_chunks = (chunks + 2) // 3
    est_time_sec = (parallel_chunks * time_per_chunk) + 50
    
    return {
        'days': days,
        'estimated_trades': f"{total_trades:,.0f}",
        'estimated_size_mb': f"{size_mb:.1f}",
        'estimated_time_min': f"{est_time_sec / 60:.1f}",
        'recommended_chunk': 'monthly' if days > 365 else ('weekly' if days > 60 else 'daily')
    }

# Example: Estimate for 2015-2025
print("📊 Download Estimate for SPY (2015-2025):")
est = estimate_download_time(datetime(2015, 1, 1), datetime(2025, 12, 31), avg_trades_per_day=1_500_000)
for key, val in est.items():
    print(f"  • {key}: {val}")

print(f"\n💡 OPTIMIZATION TIPS FOR LARGE RANGES:")
print(f"  1. Use chunk_type='monthly' for multi-year ranges")
print(f"  2. Resume capability: Already-cached dates are skipped automatically")
print(f"  3. For 10+ years, consider splitting into 2-3 downloads by year")
print(f"  4. Monitor rate limits: max_workers=2-3 is safe; don't exceed 5")
print(f"  5. Run overnight for large jobs; use caching for development")
